# Регрессия для CC50

Цель: построить и сравнить несколько моделей регрессии для прогнозирования CC50, оценить их качество и сформулировать рекомендации

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV

df = pd.read_csv('/Users/yaroslavbaev/Desktop/miphi/data/chem_data_prepared.csv')

X = df.drop(columns=['IC50, mM', 'CC50, mM', 'SI'])
y = df['CC50, mM']

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [2]:
def reg_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred) 
    rmse = np.sqrt(mse)                       
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': rmse,
        'R2': r2_score(y_true, y_pred),
    }

## Базовые линейные модели

In [3]:
ridge = Ridge(random_state=42)
ridge.fit(X_train_scaled, y_train)
ridge_val = reg_metrics(y_val, ridge.predict(X_val_scaled))

lasso = Lasso(random_state=42)
lasso.fit(X_train_scaled, y_train)
lasso_val = reg_metrics(y_val, lasso.predict(X_val_scaled))

ridge_val, lasso_val

/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.394e+06, tolerance: 2.649e+04
  model = cd_fast.enet_coordinate_descent(


({'MAE': 358.7196325327598,
  'RMSE': np.float64(674.5945096025913),
  'R2': -0.6895026525945194},
 {'MAE': 334.4298430371159,
  'RMSE': np.float64(589.2082454684574),
  'R2': -0.28887531234048525})

## RandomForestRegressor с подбором

In [4]:
rf = RandomForestRegressor(random_state=42, n_jobs=-1)

rf_params = {
    'n_estimators': [200, 400, 600],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 0.5],
}

rf_search = RandomizedSearchCV(
    rf,
    rf_params,
    n_iter=25,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    random_state=42,
    verbose=1,
)
rf_search.fit(X_train, y_train)
rf_search.best_params_, rf_search.best_score_

Fitting 3 folds for each of 25 candidates, totalling 75 fits


({'n_estimators': 200,
  'min_samples_split': 2,
  'min_samples_leaf': 2,
  'max_features': 0.5,
  'max_depth': 10},
 np.float64(-447.5333570589419))

In [5]:
rf_best = rf_search.best_estimator_
rf_val = reg_metrics(y_val, rf_best.predict(X_val))
rf_test = reg_metrics(y_test, rf_best.predict(X_test))
rf_val, rf_test

({'MAE': 281.1177950312659,
  'RMSE': np.float64(380.41690554182077),
  'R2': 0.4627296296075404},
 {'MAE': 295.9862416123589,
  'RMSE': np.float64(463.95866057270274),
  'R2': 0.5848055381004436})

## XGBRegressor с подбором

In [6]:
xgb = XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    n_estimators=500,
    n_jobs=-1,
)

xgb_params = {
    'max_depth': [3, 4, 5, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'min_child_weight': [1, 3, 5, 7],
}

xgb_search = RandomizedSearchCV(
    xgb,
    xgb_params,
    n_iter=30,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    random_state=42,
    verbose=1,
)
xgb_search.fit(X_train, y_train)
xgb_search.best_params_, xgb_search.best_score_

Fitting 3 folds for each of 30 candidates, totalling 90 fits


({'subsample': 0.6,
  'min_child_weight': 3,
  'max_depth': 6,
  'learning_rate': 0.01,
  'colsample_bytree': 0.6},
 np.float64(-429.17753248615963))

In [7]:
xgb_best = xgb_search.best_estimator_
xgb_val = reg_metrics(y_val, xgb_best.predict(X_val))
xgb_test = reg_metrics(y_test, xgb_best.predict(X_test))
xgb_val, xgb_test

({'MAE': 273.7942530640463,
  'RMSE': np.float64(372.5842521903206),
  'R2': 0.4846262870355542},
 {'MAE': 284.96429903341794,
  'RMSE': np.float64(452.8989128148232),
  'R2': 0.6043642423081317})